<a href="https://colab.research.google.com/github/franciscojsp82/chatbot-uaf-laft/blob/main/UAF_LAFT_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot RAG — Detección de Lavado de Activos y Financiamiento del Terrorismo (UAF Chile)

## Problem Statement

Se busca construir un asistente conversacional (RAG — Retrieval-Augmented Generation) que permita a oficiales de cumplimiento y sujetos obligados consultar, en lenguaje natural, la documentación oficial de la Unidad de Análisis Financiero (UAF) de Chile en materia de Lavado de Activos, Financiamiento del Terrorismo y Financiamiento de la Proliferación (LA/FT/FP).

El corpus consolidado (`LAFT_UAF_Chile.pdf`) incluye:

1. Ley N°19.913 — Crea la Unidad de Análisis Financiero.
2. Anexos N°1 y N°2 de la Circular N°62 (UAF).
3. Circular N°62 (UAF) — Instrucciones de carácter general a sujetos obligados.
4. Señales de Alerta para Prevenir el Cohecho a Funcionarios Públicos Extranjeros (UAF, 2014).
5. Guía de Señales de Alerta de LA/FT/FP — Actualización 2023 (UAF).
6. Señales de Alerta — Instituciones Públicas (UAF, 2016).
7. Catálogo de Delitos Base o Precedentes de Lavado de Activos (UAF, septiembre 2025).

El chatbot responde, con base **exclusivamente** en este corpus, preguntas sobre:

- **Señales de alerta**: dado un comportamiento u operación descrita por el usuario, ¿constituye un indicio de LA/FT/FP?
- **Obligación de informar**: ¿corresponde reportar a la UAF? ¿Qué tipo de informe (ROS, ROE, u otro) y con qué periodicidad/plazo?
- **Umbrales**: ¿qué montos o condiciones activan la obligación de reportar según el tipo de operación?
- **Sanciones**: ¿cuáles son las sanciones por lavado de activos, y por no informar o informar incorrecta/extemporáneamente?

## Sobre el motor del LLM (importante)

Versiones anteriores de este notebook usaban `llama-cpp-python` con un modelo GGUF (Mistral-7B).
Ese enfoque requiere **compilar C++ desde código fuente** al instalar, lo cual actualmente **falla en
Python 3.13** (la versión que trae Colab por defecto) por incompatibilidades del proyecto con ese
estándar de C++ — es un problema conocido, no de esta instalación en particular.

Para eliminar ese problema de raíz, este notebook usa en cambio **Hugging Face `transformers` +
`bitsandbytes`** (cuantización de 4 bits) para cargar `HuggingFaceH4/zephyr-7b-beta` (modelo
instruct basado en Mistral-7B, licencia Apache 2.0, de descarga abierta sin necesidad de token).
Estas librerías se instalan siempre desde wheels precompiladas — nunca compilan nada en tu máquina —
por lo que este tipo de falla ya no puede ocurrir.

## ⚠️ Guía rápida de ejecución (léela antes de correr el notebook)

1. **Tipo de entorno**: `Entorno de ejecución > Cambiar tipo de entorno de ejecución` → elige **GPU (T4)**.
   La cuantización de 4 bits (`bitsandbytes`) requiere GPU; sin ella el modelo igual carga pero en CPU
   sin cuantizar, lo que es *muy* lento (minutos por respuesta). El notebook detecta esto solo y te avisa.
2. **Corre la celda de instalación una sola vez.** Al final, el notebook **reinicia el proceso de
   Python automáticamente** (verás un aviso de "sesión reiniciada" — es esperado, no un error).
3. **Después del reinicio, usa `Entorno de ejecución > Ejecutar todo` una segunda vez.** Los paquetes
   ya instalados no se reinstalan, así que corre rápido hasta donde íbamos.
4. **Sube `LAFT_UAF_Chile.pdf`** al panel de archivos (📁, barra izquierda) o móntalo desde Drive.
5. Si ves errores `AttributeError`/`cannot import name` de numpy: `Reiniciar la sesión` y `Ejecutar
   todo` de nuevo. Si persiste, `Desconectar y borrar el tiempo de ejecución` para una VM limpia.

## Instalación (celda única — solo wheels precompiladas, sin compilación C++)

In [ ]:
import subprocess, os

# --- 1. Detectar si hay GPU NVIDIA disponible ---
has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"GPU detectada: {has_gpu}")

# --- 2. Instalar todo desde wheels precompiladas (nada se compila localmente) ---
# Sin -U, para no forzar upgrades masivos que choquen con paquetes preinstalados de Colab.
%pip install -q huggingface_hub pandas tiktoken pymupdf \
    langchain-text-splitters langchain-community langchain-chroma langchain-huggingface \
    chromadb sentence-transformers \
    transformers accelerate bitsandbytes

# --- 3. Guardar si hay GPU para reutilizarlo tras el reinicio ---
with open("/content/_has_gpu.flag", "w") as f:
    f.write("1" if has_gpu else "0")

# --- 4. Reinicio automático del proceso de Python ---
# Instalar/actualizar numpy, torch, etc. a mitad de sesión deja versiones mezcladas en
# memoria vs. disco (causa típica de errores "AttributeError" / "cannot import" más adelante).
# Esto reinicia el proceso de Python del kernel (NO borra la VM ni lo ya instalado).
print("\nReiniciando el entorno de Python para cargar limpio lo instalado...")
print("Cuando vuelva a conectar, corre 'Ejecutar todo' de nuevo (la instalación no se repetirá).")
os.kill(os.getpid(), 9)


GPU detectada: True


## Importación de librerías

In [1]:
#Librerías para procesar dataframes y texto
import json, os, re
import tiktoken
import pandas as pd

#Librerías para cargar datos, chunking, embeddings y bases de datos vectoriales (LangChain 1.x)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

#Librerías para cargar el LLM (transformers + bitsandbytes, sin compilación C++)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Recuperar si hay GPU (detectado en la celda de instalación)
try:
    with open("/content/_has_gpu.flag") as f:
        HAS_GPU = f.read().strip() == "1"
except FileNotFoundError:
    HAS_GPU = torch.cuda.is_available()
print(f"GPU disponible para este notebook: {HAS_GPU}")


/tmp/ipykernel_7730/1687362460.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


GPU disponible para este notebook: True


## Preparación de datos para RAG

### Carga del corpus consolidado de la UAF

Ejecuta **una** de las dos opciones siguientes para poner el PDF a disposición del notebook,
antes de la celda que verifica la ruta.

In [2]:
# Opción A: montar Google Drive (recomendado si vas a usar el notebook en varias sesiones)
# Descomenta y corre esto si guardaste el PDF en tu Drive (MyDrive/LAFT_UAF_Chile.pdf):

# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# Opción B: subir el archivo manualmente a esta sesión de Colab
# (panel de archivos a la izquierda > ícono de subir archivo > selecciona LAFT_UAF_Chile.pdf)
# No requiere código; solo asegúrate de que termine de subir antes de correr la celda siguiente.

In [4]:
# Ruta al PDF consolidado de la UAF. Ajusta según dónde hayas dejado el archivo.
POSSIBLE_PATHS = [
    "/content/LAFT_UAF_Chile.pdf",
    "/content/drive/MyDrive/LAFT_UAF_Chile.pdf",
    "LAFT_UAF_Chile.pdf",
]

uaf_pdf_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)

if uaf_pdf_path is None:
    print("No se encontró el PDF en ninguna de estas rutas:")
    for p in POSSIBLE_PATHS:
        print("  -", p)
    print("\nSube LAFT_UAF_Chile.pdf al panel de archivos, o monta tu Drive (celda de Opción A) y vuelve a correr esta celda.")
else:
    print(f"Usando PDF: {uaf_pdf_path}")

assert uaf_pdf_path is not None, "PDF no encontrado. Revisa el mensaje de arriba antes de continuar."

Usando PDF: /content/LAFT_UAF_Chile.pdf


In [5]:
pdf_loader = PyMuPDFLoader(uaf_pdf_path)
uaf_docs = pdf_loader.load()

### Exploración de los datos

#### Revisión de las primeras páginas

In [6]:
for i in range(3):
    print(f"Página : {i+1}", end="\n")
    print(uaf_docs[i].page_content, end="\n")
    print("-"*80)

Página : 1
Corpus de Documentación Oficial UAF Chile
Técnicas de detección de lavado de activos y financiamiento del terrorismo
Documento consolidado para uso en sistema de búsqueda / chatbot (RAG)
Documentos incluidos:
1. LEY N°19.913 - Crea la Unidad de Análisis Financiero y modifica diversas disposiciones en
materia de lavado y blanqueo de activos
2. ANEXOS N°1 y N°2 de la Circular N°62 (UAF)
3. CIRCULAR N°62 (UAF) - Instrucciones de carácter general a sujetos obligados
4. Señales de Alerta para Prevenir el Cohecho a Funcionarios Públicos Extranjeros (UAF, 2014)
5. Guía de Señales de Alerta de LA/FT/FP - Actualización 2023 (UAF)
6. Señales de Alerta - Instituciones Públicas (UAF, 2016)
7. Catálogo de Delitos Base o Precedentes de Lavado de Activos (UAF, septiembre 2025)
--------------------------------------------------------------------------------
Página : 2
[INICIO_DOCUMENTO_1]
LEY N°19.913 - Crea la Unidad de Análisis Financiero y modifica diversas
disposiciones en materia de la

#### Número de páginas

In [7]:
len(uaf_docs)

167

#### Documentos que componen el corpus consolidado

El PDF fue consolidado previamente uniendo 7 documentos oficiales de la UAF, delimitados con
marcadores `[INICIO_DOCUMENTO_N]` / `[FIN_DOCUMENTO_N]`. Los identificamos para tener trazabilidad
de qué documento respalda cada respuesta del chatbot.

In [8]:
doc_titles = {}
pattern = re.compile(r"\[INICIO_DOCUMENTO_(\d+)\]\s*(.*)")

for page_num, d in enumerate(uaf_docs, start=1):
    match = pattern.search(d.page_content)
    if match:
        doc_num, title_line = match.groups()
        doc_titles[int(doc_num)] = (page_num, title_line.strip()[:100])

for doc_num in sorted(doc_titles):
    page_num, title = doc_titles[doc_num]
    print(f"Documento {doc_num} (página {page_num}): {title}")

Documento 1 (página 2): LEY N°19.913 - Crea la Unidad de Análisis Financiero y modifica diversas
Documento 2 (página 34): ANEXOS N°1 y N°2 de la Circular N°62 (UAF)
Documento 3 (página 37): CIRCULAR N°62 (UAF) - Instrucciones de carácter general a sujetos
Documento 4 (página 56): Señales de Alerta para Prevenir el Cohecho a Funcionarios Públicos
Documento 5 (página 58): Guía de Señales de Alerta de LA/FT/FP - Actualización 2023 (UAF)
Documento 6 (página 126): Señales de Alerta - Instituciones Públicas (UAF, 2016)
Documento 7 (página 134): Catálogo de Delitos Base o Precedentes de Lavado de Activos (UAF,


### Chunking (segmentación del texto)

In [9]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=50
)

In [10]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [11]:
len(document_chunks)

349

In [12]:
print(document_chunks[0].page_content)

Corpus de Documentación Oficial UAF Chile
Técnicas de detección de lavado de activos y financiamiento del terrorismo
Documento consolidado para uso en sistema de búsqueda / chatbot (RAG)
Documentos incluidos:
1. LEY N°19.913 - Crea la Unidad de Análisis Financiero y modifica diversas disposiciones en
materia de lavado y blanqueo de activos
2. ANEXOS N°1 y N°2 de la Circular N°62 (UAF)
3. CIRCULAR N°62 (UAF) - Instrucciones de carácter general a sujetos obligados
4. Señales de Alerta para Prevenir el Cohecho a Funcionarios Públicos Extranjeros (UAF, 2014)
5. Guía de Señales de Alerta de LA/FT/FP - Actualización 2023 (UAF)
6. Señales de Alerta - Instituciones Públicas (UAF, 2016)
7. Catálogo de Delitos Base o Precedentes de Lavado de Activos (UAF, septiembre 2025)


In [13]:
print(document_chunks[-1].page_content)

Página 24 de 50
Página 25 de 50
Página 32 de 50
Página 33 de 50
Página 34 de 50
Página 35 de 50
Página 36 de 50
Página 37 de 50
Página 39 de 50
Página 41 de 50
Página 48 de 50
Página 49 de 50
Página 50 de 50
[FIN_DOCUMENTO_7]


### Embeddings

In [14]:
# Modelo de embeddings multilingüe (el corpus está en español)
embedding_model = HuggingFaceEmbeddings(model_name='thenlper/gte-large')

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  670MB            

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [15]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [16]:
print("Dimensión del vector de embedding: ", len(embedding_1))
len(embedding_1) == len(embedding_2)

Dimensión del vector de embedding:  1024


True

### Base de datos vectorial

In [17]:
out_dir = 'uaf_chroma_db'

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [18]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [19]:
vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)

In [20]:
# Prueba rápida de búsqueda por similitud
vectorstore.similarity_search("fraccionamiento de depósitos en efectivo", k=3)

[Document(id='10d8e85f-5988-42e5-b04b-e129a7e5f44e', metadata={'page': 84, 'source': '/content/LAFT_UAF_Chile.pdf', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'creationdate': '2026-09-04T15:38:28+00:00', 'modDate': "D:20260904153828+00'00'", 'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'format': 'PDF 1.4', 'file_path': '/content/LAFT_UAF_Chile.pdf', 'creationDate': "D:20260904153828+00'00'", 'trapped': '', 'moddate': '2026-09-04T15:38:28+00:00', 'total_pages': 167, 'keywords': ''}, page_content='incremento de dinero abonado por medio de depósitos en efectivo, de terceras personas\ny endosados a nombre del titular de la cuenta.\nlos cuales son girados en efectivo por caja o cajero automático\ndurante el mismo día, o en un corto periodo, desconociéndose su 9.44 Cliente recibe\ndepósitos bancarios de su empleador,\ndestino final. adicionales a su remuneración, generalmente al día siguiente del\npago de la misma, sin moti

### Retriever

In [21]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [22]:
rel_docs = retriever.invoke("¿Qué es el Reporte de Operaciones Sospechosas?")
rel_docs

[Document(id='11197132-e690-4aad-89fa-350417282f5e', metadata={'subject': '(unspecified)', 'modDate': "D:20260904153828+00'00'", 'title': '(anonymous)', 'file_path': '/content/LAFT_UAF_Chile.pdf', 'moddate': '2026-09-04T15:38:28+00:00', 'trapped': '', 'author': '(anonymous)', 'keywords': '', 'page': 59, 'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-04T15:38:28+00:00', 'source': '/content/LAFT_UAF_Chile.pdf', 'total_pages': 167, 'format': 'PDF 1.4', 'creationDate': "D:20260904153828+00'00'"}, page_content='clientes o características de ciertos actos que\npueden develar estar ante la presencia u ocurrencia de una operación sospechosa de\nLA/FT. Por tanto, ayudan a distinguir hechos,\nsituaciones, transacciones, eventos, cuantías o indicadores financieros que la\nexperiencia nacional e internacional han identificado como\nelementos de juicio, a partir de los cuales se puede inferir la posible existencia de\nun hecho o situación qu

## Descarga y carga del LLM

Se usa `HuggingFaceH4/zephyr-7b-beta` (Mistral-7B fine-tuneado, Apache 2.0, sin necesidad de token
de Hugging Face) vía `transformers`. Con GPU se carga cuantizado a 4 bits (`bitsandbytes`); sin GPU,
carga en CPU sin cuantizar (funcional, pero mucho más lento).

In [23]:
MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"
N_CTX = 8192  # ventana de contexto nativa del modelo base (Mistral-7B)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if HAS_GPU:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    print("Modelo cargado en GPU, cuantizado a 4 bits.")
else:
    print("⚠️ Sin GPU: cargando en CPU sin cuantizar. Esto puede tardar varios minutos por respuesta.")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

model.eval()

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Modelo cargado en GPU, cuantizado a 4 bits.


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

### Función de generación (wrapper sobre `model.generate`)

In [24]:
def llm_generate(system_message, user_message, max_tokens=400,
                  temperature=0, top_p=0.95, top_k=50):
    """
    Arma el prompt con la plantilla de chat propia del modelo (system + user) y genera texto.
    temperature=0 -> generación determinista (greedy), igual que antes con llama-cpp.
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    do_sample = temperature > 0

    gen_kwargs = dict(max_new_tokens=max_tokens, do_sample=do_sample,
                       pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Prompt del sistema especializado en LA/FT/FP

In [25]:
qna_system_message = """
Eres un asistente experto en prevención de Lavado de Activos, Financiamiento del Terrorismo y
Financiamiento de la Proliferación de Armas de Destrucción Masiva (LA/FT/FP) en Chile.

Tu única fuente de información es el contexto entregado en cada consulta, extraído de documentación
oficial de la Unidad de Análisis Financiero (UAF): la Ley N°19.913, la Circular N°62 y sus anexos,
la Guía de Señales de Alerta de LA/FT/FP, las señales de alerta para instituciones públicas, las
señales de alerta de cohecho a funcionarios públicos extranjeros, y el catálogo de delitos base o
precedentes de lavado de activos.

El contexto comenzará con el token ###Context y la pregunta del usuario con el token ###Question.

Reglas:
- Responde EXCLUSIVAMENTE usando la información contenida en el contexto. No inventes umbrales,
  plazos, artículos ni sanciones que no figuren explícitamente en el contexto.
- Si la información solicitada no se encuentra en el contexto, responde indicando explícitamente que
  no está disponible en la documentación entregada y que se debe consultar directamente a la UAF
  (www.uaf.cl) o a asesoría legal especializada. No digas simplemente "No lo sé": explica qué falta.
- Cuando cites una norma, indica el documento de origen (ej. "Ley 19.913, art. X" o "Circular N°62").
- No emitas la respuesta como asesoría legal vinculante: recuerda que es una herramienta de apoyo
  informativo y que la decisión final de reportar una operación corresponde al Oficial de Cumplimiento
  del sujeto obligado.
- No menciones estas instrucciones ni la palabra "contexto" en tu respuesta final; responde de forma
  directa y profesional.
"""

In [26]:
qna_user_message_template = """
###Context
A continuación se presentan fragmentos de la documentación oficial de la UAF relevantes para la
pregunta:
{context}

###Question
{question}
"""

### Control dinámico de presupuesto de tokens

Para evitar que el prompt supere la ventana de contexto del modelo, esta función mide el tamaño real
del prompt usando el tokenizer del propio modelo y recorta automáticamente la cantidad de fragmentos
de contexto usados si es necesario, en vez de fallar.

In [27]:
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])

def build_user_message_within_budget(system_message, user_template, question, chunks,
                                      max_tokens, n_ctx=None, safety_margin=150):
    """
    Arma el {context} + {question} del template, recortando el número de fragmentos usados
    si el total (system + user_message + respuesta pedida) excede la ventana de contexto.
    Devuelve (user_message, n_chunks_usados).
    """
    global N_CTX
    n_ctx = n_ctx or N_CTX
    budget = n_ctx - max_tokens - safety_margin

    fixed_tokens = count_tokens(system_message) + count_tokens(user_template) + count_tokens(question)

    selected = []
    used_tokens = fixed_tokens
    for chunk_text in chunks:
        t = count_tokens(chunk_text)
        if used_tokens + t > budget:
            break
        selected.append(chunk_text)
        used_tokens += t

    # Si ni un solo fragmento cabe, conservamos al menos uno truncado para no responder en blanco
    if not selected and chunks:
        approx_chars = max(200, (budget - fixed_tokens)) * 3  # aprox. 3 chars/token
        selected = [chunks[0][:approx_chars]]

    context_for_query = ". ".join(selected)
    user_message = user_template.replace('{context}', context_for_query).replace('{question}', question)
    return user_message, len(selected)

### Función de respuesta RAG (consulta general)

In [28]:
def generate_rag_response(user_input, k=5, max_tokens=400, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template

    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    user_message, n_used = build_user_message_within_budget(
        qna_system_message, qna_user_message_template, user_input, context_list, max_tokens
    )
    if n_used < len(context_list):
        print(f"[Aviso] Se usaron {n_used}/{len(context_list)} fragmentos para caber en la ventana de contexto.")

    try:
        response = llm_generate(qna_system_message, user_message, max_tokens, temperature, top_p, top_k)
    except Exception as e:
        response = f'Lo siento, ocurrió el siguiente error: \n{e}'

    return response

## Función especializada — Análisis de cumplimiento

El chatbot ofrece un modo especializado para analizar el cumplimiento
de la Ley contra LA/FT y el **comportamiento u operación de un cliente**,
a través de las siguientes consultas:

1. ¿Que instituciones deben reportar?.
2. ¿Qué reportes debo enviar?.
3. ¿Qué sanciones y normativa está asociada a los siguientes incumplimientos?.
4. Señales de Alerta LA/FT.
5. Pregunta general (cualquier otra consulta sobre LA/FT/FP).

In [29]:
behavior_analysis_system_message = qna_system_message + """

Además, el ###Question describirá un comportamiento u operación de un cliente. Tu respuesta DEBE
seguir EXACTAMENTE este formato, usando la información del contexto:

1. Señales de alerta detectadas: lista las señales de alerta de LA/FT/FP que coinciden con el
   comportamiento descrito, citando el documento UAF de origen. Si no se detecta ninguna señal de
   alerta relevante en el contexto, indícalo explícitamente.
2. ¿Corresponde reportar?: Sí / No / Con precaución (evaluar más antecedentes), y por qué.
3. Tipo de reporte y periodicidad: si corresponde, indica el tipo de informe (por ejemplo, Reporte de
   Operación Sospechosa - ROS, Reporte de Operación en Efectivo - ROE, u otro) y el plazo o
   periodicidad establecida en el contexto.
4. Umbral aplicable: monto o condición umbral, si el contexto lo especifica para el tipo de operación
   descrito. Si no hay umbral aplicable (ej. el ROS no tiene umbral mínimo), indícalo.
5. Sanciones aplicables: sanciones por lavado de activos y/o por no informar, informar tardíamente o
   de forma incorrecta, según lo que indique el contexto.
"""

In [30]:
def analyze_customer_behavior(descripcion_comportamiento, k=6, max_tokens=900,
                               temperature=0, top_p=0.95, top_k=50):
    """
    Evalúa un comportamiento/operación de cliente frente a las señales de alerta,
    obligaciones de reporte, umbrales y sanciones descritas en la documentación UAF.
    """
    global behavior_analysis_system_message, qna_user_message_template

    relevant_document_chunks = vectorstore.similarity_search(descripcion_comportamiento, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    user_message, n_used = build_user_message_within_budget(
        behavior_analysis_system_message, qna_user_message_template,
        descripcion_comportamiento, context_list, max_tokens
    )
    if n_used < len(context_list):
        print(f"[Aviso] Se usaron {n_used}/{len(context_list)} fragmentos para caber en la ventana de contexto.")

    try:
        response = llm_generate(behavior_analysis_system_message, user_message, max_tokens,
                                 temperature, top_p, top_k)
    except Exception as e:
        response = f'Lo siento, ocurrió el siguiente error: \n{e}'

    return response

In [32]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [33]:
# ============================================================
# PREGUNTA 1 — ¿Debo informar a la UAF según el tipo de entidad?
# (determinístico, Art. 3° Ley N°19.913 — no usa el LLM)
# ============================================================
SUJETOS_OBLIGADOS = [
    "Bancos e instituciones financieras",
    "Representaciones de bancos extranjeros",
    "Cooperativas de ahorro y crédito",
    "Empresas de factoraje",
    "Empresas de arrendamiento financiero (leasing)",
    "Empresas de securitización",
    "Bolsas de valores y bolsas de productos",
    "Corredores de bolsa / agentes de valores",
    "Administradoras generales de fondos y de fondos de inversión privados",
    "Administradores de fondos mutuos",
    "Operadores de mercados de futuro y opciones",
    "Empresas de depósito de valores (Ley N°18.876)",
    "Plataformas de financiamiento colectivo, custodia o intermediación de instrumentos financieros, "
    "o iniciación de pagos (inscritas en registros CMF)",
    "Compañías de seguros",
    "Administradoras de fondos de pensiones (AFP)",
    "Emisoras/operadoras de tarjetas de crédito o de pago con provisión de fondos",
    "Empresas de transferencia y transporte de valores y dinero",
    "Casas de cambio y entidades facultadas para recibir moneda extranjera",
    "Casinos, salas de juego e hipódromos",
    "Titulares de permisos de juegos de azar en naves mercantes mayores (fines turísticos)",
    "Corredores de propiedades y empresas de gestión inmobiliaria",
    "Notarios",
    "Conservadores",
    "Casas de remate y martillo",
    "Comerciantes de metales preciosos, joyas y piedras preciosas",
    "Automotoras y comercializadoras de vehículos nuevos o usados",
    "Empresas de arriendo de vehículos",
    "Personas naturales o jurídicas dedicadas a la compraventa de equinos de raza pura",
    "Fabricantes o vendedores de armas",
    "Clubes de tiro, caza y pesca",
    "Agentes de aduana",
    "Administradoras y usuarios de zonas francas",
    "Organizaciones deportivas profesionales (Ley N°20.019)",
    "Otra entidad fiscalizada por la CMF, inscrita voluntariamente en el registro (Art. 40)",
]

EJEMPLOS_NO_OBLIGADOS = [
    "Establecimiento educacional (colegio, escuela, universidad)",
    "Organización sin fines de lucro / fundación (no inscrita voluntariamente en CMF)",
    "Restaurante o servicio de alimentación",
    "Comercio minorista general (no vehículos, no metales/joyas preciosas)",
    "Clínica, hospital u otro prestador de salud",
    "Estudio jurídico o consultora (no financiera)",
    "Empresa de tecnología / software",
    "Empresa de manufactura (no armas)",
    "Servicio público no fiscalizado por la CMF",
    "Otro (no corresponde a ninguna categoría del Art. 3°)",
]

_opciones_pj = (
    [(f"✅ {item}", ("si", item)) for item in SUJETOS_OBLIGADOS]
    + [("──────── Ejemplos que NO están obligados ────────", ("separador", None))]
    + [(f"❌ {item}", ("no", item)) for item in EJEMPLOS_NO_OBLIGADOS]
)

persona_juridica_selector = widgets.Dropdown(
    options=_opciones_pj,
    description='Tipo de entidad:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)

boton_verificar_pj = widgets.Button(
    description='Verificar',
    button_style='primary',
    icon='check',
)

salida_pj = widgets.Output()

def _on_verificar_pj(b):
    with salida_pj:
        clear_output()
        estado, item = persona_juridica_selector.value
        if estado == 'separador':
            print("Por favor selecciona un tipo de entidad (no el separador).")
            return
        if estado == 'si':
            mensaje = (
                f"**Sí, usted debe informar a la UAF.**\n\n"
                f"Su entidad corresponde a la categoría *\"{item}\"*, expresamente listada como sujeto "
                f"obligado en el Artículo 3° de la Ley N°19.913."
            )
        else:
            mensaje = (
                f"**No, usted no debe informar a la UAF.**\n\n"
                f"La categoría *\"{item}\"* no se encuentra en el listado de sujetos obligados del "
                f"Artículo 3° de la Ley N°19.913 (ejemplo ilustrativo). Excepción: si su entidad está "
                f"fiscalizada por la Comisión para el Mercado Financiero (CMF) y se inscribió voluntariamente "
                f"en el Registro del Artículo 40, sí correspondería informar."
            )
        display(Markdown(mensaje))

boton_verificar_pj.on_click(_on_verificar_pj)


# ============================================================
# PREGUNTAS 2-5 — Formulario de texto libre guiado
# ============================================================
modo_selector = widgets.RadioButtons(
    options=[
        ('2. ¿Qué reportes debo enviar?', 'reportes'),
        ('3. ¿Qué sanciones y normativa está asociada a los siguientes incumplimientos?', 'sanciones'),
        ('4. Señales de Alerta LA/FT', 'comportamiento'),
        ('5. Pregunta general (cualquier otra consulta sobre LA/FT/FP)', 'general'),
    ],
    value='reportes',
    description='Modo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)

PLACEHOLDERS = {
    'general': 'Escribe tu pregunta sobre LA/FT/FP...',
    'reportes': 'Describe tu situación u operación (tipo de institución, tipo de operación, monto, etc.) '
                'para precisar qué reportes debes enviar...',
    'sanciones': 'Describe el/los incumplimientos específicos (ej: no informar a tiempo, informar de '
                 'forma incorrecta, no reportar una operación sospechosa, lavado de activos consumado, etc.)...',
    'comportamiento': 'Describe el comportamiento/operación del cliente...',
}

pregunta_input = widgets.Textarea(
    value='',
    placeholder=PLACEHOLDERS['reportes'],
    layout=widgets.Layout(width='95%', height='120px'),
)

def _on_modo_change(change):
    pregunta_input.placeholder = PLACEHOLDERS.get(change['new'], '')

modo_selector.observe(_on_modo_change, names='value')

boton_consultar = widgets.Button(
    description='Consultar',
    button_style='primary',
    icon='search',
)

boton_limpiar = widgets.Button(
    description='Limpiar',
    icon='eraser',
)

salida = widgets.Output()

def _on_consultar(b):
    with salida:
        clear_output()
        modo = modo_selector.value
        pregunta = pregunta_input.value.strip()

        if not pregunta:
            print("Por favor escribe una pregunta o descripción antes de consultar.")
            return

        print("Consultando... esto puede tardar unos segundos (o más si el runtime es CPU).")
        try:
            if modo == 'comportamiento':
                respuesta = analyze_customer_behavior(pregunta, max_tokens=900)
            elif modo == 'reportes':
                pregunta_final = (
                    f"En relación a la siguiente situación: {pregunta}. "
                    "¿Qué reportes debo enviar a la UAF, con qué periodicidad y qué contenido deben incluir?"
                )
                respuesta = generate_rag_response(pregunta_final, max_tokens=700)
            elif modo == 'sanciones':
                pregunta_final = (
                    f"¿Qué sanciones y normativa están asociadas a los siguientes incumplimientos: {pregunta}?"
                )
                respuesta = generate_rag_response(pregunta_final, max_tokens=700)
            else:  # 'general'
                respuesta = generate_rag_response(pregunta, max_tokens=700)
        except Exception as e:
            respuesta = f"Ocurrió un error al generar la respuesta: {e}"

        clear_output()
        display(Markdown(f"**Pregunta / caso:**\n\n{pregunta}\n\n---\n\n**Respuesta:**\n\n{respuesta}"))

def _on_limpiar(b):
    pregunta_input.value = ''
    with salida:
        clear_output()

boton_consultar.on_click(_on_consultar)
boton_limpiar.on_click(_on_limpiar)


# ============================================================
# Mostrar el formulario completo (Pregunta 1 + Preguntas 2-5)
# ============================================================
display(widgets.VBox([
    widgets.HTML("<h3>💬 Consulta al Chatbot UAF — Prevención de LA/FT/FP</h3>"),
    widgets.HTML("<b>1. ¿Debo informar a la UAF? (según tipo de entidad)</b>"),
    persona_juridica_selector,
    boton_verificar_pj,
    salida_pj,
    widgets.HTML("<hr>"),
    modo_selector,
    pregunta_input,
    widgets.HBox([boton_consultar, boton_limpiar]),
    salida,
]))

## Evaluación de la calidad de las respuestas (Groundedness y Relevance)

In [34]:
groundedness_rater_system_message = """
Se te encargará evaluar respuestas generadas por un sistema de IA a preguntas formuladas por
usuarios en el dominio de prevención de lavado de activos y financiamiento del terrorismo.
Se te presentará una pregunta, el contexto normativo usado por el sistema para generar la respuesta,
y la respuesta generada. La pregunta comenzará con ###Question, el contexto con ###Context y la
respuesta con ###Answer.

Criterio de evaluación:
La tarea consiste en juzgar en qué medida se cumple la métrica.
1 - La métrica no se cumple en absoluto
2 - La métrica se cumple solo en una medida limitada
3 - La métrica se cumple en buena medida
4 - La métrica se cumple mayormente
5 - La métrica se cumple completamente

Métrica:
La respuesta debe derivarse únicamente de la información presentada en el contexto, sin inventar
umbrales, plazos, artículos legales ni sanciones que no figuren en él.

Instrucciones:
1. Primero, escribe los pasos necesarios para evaluar la respuesta según la métrica.
2. Da una explicación paso a paso de si la respuesta cumple la métrica, considerando la pregunta y el
   contexto como entrada.
3. Luego, evalúa el grado de cumplimiento de la métrica.
4. Usa la información anterior para calificar la respuesta según el criterio de evaluación y asigna un
   puntaje.
"""

In [35]:
relevance_rater_system_message = """
Se te encargará evaluar respuestas generadas por un sistema de IA a preguntas formuladas por
usuarios en el dominio de prevención de lavado de activos y financiamiento del terrorismo.
Se te presentará una pregunta, el contexto normativo usado por el sistema para generar la respuesta,
y la respuesta generada. La pregunta comenzará con ###Question, el contexto con ###Context y la
respuesta con ###Answer.

Criterio de evaluación:
La tarea consiste en juzgar en qué medida se cumple la métrica.
1 - La métrica no se cumple en absoluto
2 - La métrica se cumple solo en una medida limitada
3 - La métrica se cumple en buena medida
4 - La métrica se cumple mayormente
5 - La métrica se cumple completamente

Métrica:
La relevancia mide qué tan bien la respuesta aborda los aspectos principales de la pregunta (señal
de alerta aplicable, obligación de reportar, tipo/periodicidad del informe, umbral y/o sanciones,
según corresponda), con base en el contexto.

Instrucciones:
1. Primero, escribe los pasos necesarios para evaluar la respuesta según la métrica.
2. Da una explicación paso a paso de si la respuesta cumple la métrica, considerando la pregunta y el
   contexto como entrada.
3. Luego, evalúa el grado de cumplimiento de la métrica.
4. Usa la información anterior para calificar la respuesta según el criterio de evaluación y asigna un
   puntaje.
"""

In [36]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

### Función de evaluación

In [37]:
def generate_ground_relevance_response(user_input, k=5, max_tokens=350,
                                        temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template, user_message_template

    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    answer = generate_rag_response(user_input, k=k, max_tokens=max_tokens,
                                    temperature=temperature, top_p=top_p, top_k=top_k)

    eval_user_message = user_message_template.format(
        context=context_for_query, question=user_input, answer=answer
    )

    ground_msg, _ = build_user_message_within_budget(
        groundedness_rater_system_message, "{context}{question}", "", [eval_user_message], max_tokens
    )
    rel_msg, _ = build_user_message_within_budget(
        relevance_rater_system_message, "{context}{question}", "", [eval_user_message], max_tokens
    )

    ground_response = llm_generate(groundedness_rater_system_message, ground_msg, max_tokens,
                                    temperature, top_p, top_k)
    rel_response = llm_generate(relevance_rater_system_message, rel_msg, max_tokens,
                                 temperature, top_p, top_k)

    return answer, ground_response, rel_response

#### Ejemplo de evaluación: umbral de reporte en efectivo

In [38]:
user_input = "¿Cuál es el umbral, en UF o pesos, a partir del cual una operación en efectivo debe reportarse a la UAF?"
answer, ground, rel = generate_ground_relevance_response(user_input)
print("RESPUESTA:\n", answer, end="\n\n")
print("GROUNDEDNESS:\n", ground, end="\n\n")
print("RELEVANCE:\n", rel)

RESPUESTA:
 Según la información contenida en el contexto, las sujetos obligados deben informar a la UAF todas las operaciones en efectivo superiores a los USD 10.000, o su equivalente en pesos chilenos, según el valor del dólar observado el día en que se realizó la operación, a través de un Reporte de Operaciones en Efectivo (ROE), por los medios e instrucciones que defina la UAF. Por lo tanto, el umbral para reportar una operación en efectivo a la UAF es de USD 10.000 o su equivalente en pesos chilenos.

GROUNDEDNESS:
 La métrica se cumple completamente en este caso, ya que la respuesta deriva únicamente de la información presentada en el contexto y no se inventa umbrales, plazos, artículos legales ni sanciones que no figuren en él. Además, se proporciona una explicación clara de cómo se cumple la métrica y se evalúa la respuesta en base a ella.

La evaluación de la respuesta se realiza según la siguiente escala de calificación:

1 - La métrica no se cumple en absoluto
2 - La métrica

#### Ejemplo de evaluación: sanciones por no informar

In [39]:
user_input = "¿Qué sanciones existen para un sujeto obligado que no informa una operación sospechosa a la UAF, o que la informa fuera de plazo o de forma incorrecta?"
answer, ground, rel = generate_ground_relevance_response(user_input)
print("RESPUESTA:\n", answer, end="\n\n")
print("GROUNDEDNESS:\n", ground, end="\n\n")
print("RELEVANCE:\n", rel)

RESPUESTA:
 Según la Ley N°19.913 y las Circulares UAF, existen sanciones administrativas para un sujeto obligado que no informa una operación sospechosa a la UAF, o que la informa fuera de plazo o de forma incorrecta. Estas sanciones se establecen en el artículo único N°2 de la Ley 20.119 y pueden incluir multas, suspensión o cancelación de la inscripción en el Registro de la UAF, y la prohibición de realizar actividades financieras sin la autorización previa de la Comisión para el Mercado Financiero. La magnitud de la sanción dependerá de la gravedad de la falta y se decidirá en función de la situación concreta. El sujeto obligado deberá ser notificado de la sanción y tendrá la posibilidad de presentar recursos contra ella.

GROUNDEDNESS:
 La métrica establece que la respuesta debe derivarse únicamente de la información presentada en el contexto, sin inventar umbrales, plazos, artículos legales ni sanciones que no figuren en él.

Se evalúa la métrica en cuatro niveles:

1. La métrica

## Conclusiones y Recomendaciones

**Conclusiones:**
*   El enfoque híbrido final: determinístico para la Pregunta 1 (sin margen de error, basado en texto legal literal) + RAG guiado para las Preguntas 2-5 (adaptado a cada caso específico, no genérico).
*   El cambio de arquitectura de llama-cpp-python a transformers+bitsandbytes y por qué.
*   El control de tokens que evita fallos por exceso de contexto.
*   La decisión deliberada de no tener preguntas fijas de respuesta única (por la variabilidad real de cada caso).

**Recomendaciones:**
*   Precisión normativa (chunking por artículo).
*   Trazabilidad/logging, especialmente para la Pregunta 1 por su peso legal.
*   Actualización periódica del corpus y del listado de sujetos obligados si cambia la ley.
*   Validación humana como salvaguarda.
*   Migración a infraestructura persistente si el uso crece más allá de pruebas personales.
*   Cómo ampliar SUJETOS_OBLIGADOS a futuro sin tocar el modelo.